# 📊 Oracle ↔ PostgresPro Data Comparator
Модульный скрипт для потокового сравнения больших таблиц (5M+ строк).
Поддерживает разные схемы, опциональные фильтры, авто-поиск PK и визуализацию.
**Архитектура:** Подключение → Метаданные → Keyset Pagination → Инкрементальное сравнение → Dashboard.

In [ ]:
import logging as _log  # Импортируем модуль структурированного логирования
import sys as _sys  # Импортируем системные утилиты для вывода в stdout
import numpy as _np  # Импортируем библиотеку быстрых числовых вычислений
import pandas as _pd  # Импортируем библиотеку табличных вычислений
import oracledb as _oracledb  # Импортируем современный тонкий драйвер Oracle
import psycopg as _psycopg  # Импортируем современный драйвер PostgreSQL v3
import seaborn as _sns  # Импортируем библиотеку статистической визуализации
import matplotlib.pyplot as _plt  # Импортируем базовый движок отрисовки графиков
from typing import Dict, List, Optional, Any, Generator as _TypeHints  # Импортируем аннотации типов
from contextlib import contextmanager as _ctx  # Импортируем декоратор контекстных менеджеров
from tqdm.notebook import tqdm as _tqdm  # Импортируем прогресс-бар для Jupyter
import warnings as _warn  # Импортируем модуль управления предупреждениями
_warn.filterwarnings("ignore")  # Отключаем некритичные алерты внешних библиотек для чистоты вывода

In [ ]:
_logger = _log.getLogger("db_comparator")  # Создаём изолированный экземпляр логгера
_log.basicConfig(level=_log.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s", stream=_sys.stdout)  # Настраиваем формат и уровень вывода логов
class DBCompError(Exception): pass  # Определяем базовое исключение домена сравнения
class ConnError(DBCompError): pass  # Определяем исключение сетевых или авторизационных сбоев
class SchemaError(DBCompError): pass  # Определяем исключение рассинхрона метаданных

In [ ]:
@_ctx  # Декорируем функцию как управляемый ресурс
def _ora_conn(cfg: Dict[str, str]):  # Фабрика безопасного соединения с Oracle
    conn = _oracledb.connect(user=cfg["user"], password=cfg["password"], dsn=cfg.get("dsn"))  # Инициализируем соединение в thin-режиме
    conn.call_timeout = 60000  # Устанавливаем таймаут выполнения запросов в 60 секунд
    try: yield conn  # Передаём активное соединение в контекст вызывающего кода
    except Exception as e: raise ConnError(f"Oracle connection failed: {e}") from e  # Оборачиваем ошибки в доменное исключение
    finally: conn.close()  # Гарантированно закрываем сессию при выходе из контекста

@_ctx  # Декорируем функцию как управляемый ресурс
def _pg_conn(cfg: Dict[str, str]):  # Фабрика безопасного соединения с PostgresPro
    conn = _psycopg.connect(host=cfg["host"], port=cfg["port"], dbname=cfg["dbname"], user=cfg["user"], password=cfg["password"])  # Открываем TCP-соединение v3
    conn.autocommit = True  # Включаем автокоммит для оптимизации транзакций чтения
    try: yield conn  # Возвращаем объект соединения в блок with
    except Exception as e: raise ConnError(f"Postgres connection failed: {e}") from e  # Пробрасываем ошибку подключения
    finally: conn.close()  # Закрываем соединение для освобождения серверных ресурсов

In [ ]:
def _resolve_pk(conn, db: str, schema: str, table: str) -> str:  # Функция автоматического поиска первичного ключа
    cur = conn.cursor()  # Создаём курсор для выполнения мета-запроса
    try:  # Защищаем блок от синтаксических ошибок и проблем с правами
        if db == "ora":  # Ветка для Oracle
            sql = "SELECT cols.column_name FROM all_constraints c JOIN all_cons_columns cols ON c.constraint_name = cols.constraint_name WHERE c.constraint_type = 'P' AND c.owner = :1 AND c.table_name = :2"  # Формируем запрос к словарю данных Oracle
            cur.execute(sql, (schema.upper(), table.upper()))  # Выполняем параметризированный запрос
        else:  # Ветка для PostgresPro
            sql = "SELECT kcu.column_name FROM information_schema.table_constraints tc JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema = %s AND tc.table_name = %s"  # Формируем запрос к стандартному каталогу SQL
            cur.execute(sql, (schema, table))  # Выполняем запрос с параметрами
        res = cur.fetchone()  # Извлекаем первую строку результата
        if not res: raise SchemaError(f"Primary key not found in {db}.{schema}.{table}. Provide key_col explicitly.")  # Бросаем ошибку если PK отсутствует
        return res[0]  # Возвращаем имя колонки первичного ключа
    finally: cur.close()  # Освобождаем ресурсы курсора независимо от результата

def _fetch_meta(conn, db: str, schema: str, table: str) -> List[tuple]:  # Функция сбора структурной информации таблицы
    cur = conn.cursor()  # Инициализируем рабочий курсор
    try:  # Открываем защищённый блок выполнения
        if db == "ora": cur.execute("SELECT column_name, data_type, nullable, data_length FROM all_tab_columns WHERE owner = :1 AND table_name = :2 ORDER BY column_id", (schema.upper(), table.upper()))  # Запрос метаданных Oracle
        else: cur.execute("SELECT column_name, data_type, is_nullable, character_maximum_length FROM information_schema.columns WHERE table_schema = %s AND table_name = %s ORDER BY ordinal_position", (schema, table))  # Запрос метаданных Postgres
        return [r for r in cur.fetchall()]  # Возвращаем список кортежей описания колонок
    finally: cur.close()  # Закрываем курсор для предотвращения утечек памяти

In [ ]:
def _stream_chunks(conn, db: str, schema: str, table: str, key: str, chunk: int = 100000, filters: Optional[Dict] = None) -> Generator[_pd.DataFrame, None, None]:  # Генератор потоковой выгрузки с защитой от OOM
    cur = conn.cursor()  # Создаём рабочий курсор для итеративного чтения
    if db == "ora": cur.arraysize = chunk  # Настраиваем размер сетевого буфера Oracle
    else: cur.execute("SET statement_timeout = '60000'")  # Устанавливаем таймаут выполнения запросов для Postgres
    last_val = None  # Инициализируем переменную состояния курсорной пагинации
    where = "WHERE 1=1"  # Устанавливаем безопасную базовую заглушку предиката
    params = []  # Создаём пустой список для bind-переменных
    if filters:  # Проверяем наличие пользовательских условий фильтрации
        for k, v in filters.items():  # Итерируемся по словарю фильтров
            if db == "pg": where += f" AND {k} = %s"  # Добавляем условие для PostgresPro
            else: where += f" AND {k} = :{len(params)+1}"  # Добавляем условие для Oracle
            params.append(v)  # Сохраняем значение для безопасной параметризации
    try:  # Запускаем защищённый цикл выборки
        while True:  # Бесконечный цикл до полного исчерпания данных
            key_cond = f" AND {key} > :lk" if db == "ora" else f" AND {key} > %s"  # Формируем условие курсора для следующей итерации
            final_where = where + (key_cond if last_val is not None else "")  # Объединяем фильтры и пагинацию в единый предикат
            if db == "ora":  # Ветка генерации SQL для Oracle
                qry = f"SELECT * FROM {schema}.{table} {final_where} ORDER BY {key} FETCH NEXT :lim ROWS ONLY"  # Формируем запрос с курсорным лимитом
                exec_p = {**{f":{i+1}": p for i, p in enumerate(params)}, ":lim": chunk}  # Собираем единый словарь параметров
                if last_val is not None: exec_p[":lk"] = last_val  # Добавляем состояние курсора в параметры
                cur.execute(qry, exec_p)  # Выполняем параметризированный запрос
            else:  # Ветка генерации SQL для PostgresPro
                qry = f"SELECT * FROM {schema}.{table} {final_where} ORDER BY {key} LIMIT %s"  # Формируем запрос с LIMIT
                exec_p = params + [chunk] if last_val is None else params + [last_val, chunk]  # Собираем массив параметров с учётом состояния
                cur.execute(qry, exec_p)  # Выполняем запрос
            batch = cur.fetchall()  # Извлекаем пакет строк из буфера курсора
            if not batch: break  # Прерываем цикл при получении пустого результата
            df = _pd.DataFrame(batch)  # Конвертируем сырые данные в DataFrame
            last_val = df[key].iloc[-1]  # Запоминаем последнее значение ключа для следующей итерации
            yield df  # Возвращаем порцию данных вызывающему коду
            del batch, df  # Явно удаляем ссылки для освобождения памяти GC
    finally: cur.close()  # Гарантируем закрытие курсора при любом исходе

In [ ]:
class CompStrategy:  # Класс инкапсуляции правил сравнения (паттерн Стратегия)
    def __init__(self, tol: float = 1e-6, case: bool = True):  # Конструктор с настройками точности и чувствительности
        self.tol, self.case = tol, case  # Сохраняем параметры в атрибутах экземпляра
    def run(self, d1: _pd.DataFrame, d2: _pd.DataFrame, key: str, cache: Dict) -> Dict[str, Any]:  # Метод построчного сравнения чанков
        if d1.empty or d2.empty: return {"mismatches": 0, "diffs": {}, "cnt": 0}  # Выполняем быстрый выход для пустых пакетов
        common = d1.columns.intersection(d2.columns).tolist()  # Находим общие колонки для сравнения
        diffs = {c: 0 for c in common}  # Инициализируем словарь счётчиков расхождений нулями
        total = max(len(d1), len(d2))  # Определяем размер окна сравнения
        for c in common:  # Проходим по каждой общей колонке
            v1, v2 = d1[c].values, d2[c].values  # Извлекаем данные в нативные numpy-массивы
            if _np.issubdtype(v1.dtype, _np.number):  # Проверка на числовой тип данных
                mask = ~_np.isclose(v1.astype(float), v2.astype(float), atol=self.tol, equal_nan=True)  # Вычисляем маску отклонений чисел с учётом точности
            else: mask = (v1.astype(str) != v2.astype(str)) if self.case else (v1.astype(str).str.lower() != v2.astype(str).str.lower())  # Сравниваем строки с учётом или без учёта регистра
            diffs[c] += int(_np.count_nonzero(mask[:total]))  # Агрегируем количество несовпадений по колонке
            stats = d1[c].describe()  # Рассчитываем базовую статистику для текущего чанка
            if _np.issubdtype(v1.dtype, _np.number): cache[c] = {"mean": stats.get("mean", 0), "std": stats.get("std", 0), "min": stats.get("min", 0), "max": stats.get("max", 0)}  # Сохраняем метрики числовых колонок
            else: cache[f"{c}_top"] = dict(d1[c].value_counts().head(3))  # Сохраняем топ-3 частот для категориальных колонок
        return {"total": total, "diffs": diffs}  # Возвращаем результаты пакета для агрегации

In [ ]:
def _draw_dashboard(cnts: Dict, matrix: _pd.DataFrame, stats: Dict):  # Функция построения аналитического дашборда
    _plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "figure.figsize": (14, 10)})  # Настраиваем глобальные параметры отображения графиков
    fig, ax = _plt.subplots(2, 2, constrained_layout=True)  # Создаём сетку 2x2 с автоматическим выравниванием
    _sns.barplot(x=["Oracle", "PostgresPro"], y=[cnts.get("ora", 0), cnts.get("pg", 0)], ax=ax[0,0], palette="muted")  # Строим столбчатую диаграмму объёмов данных
    ax[0,0].set_title("Row Count")  # Устанавливаем заголовок первого графика
    if not matrix.empty: _sns.heatmap(matrix, annot=True, cmap="RdYlGn", fmt=".1f", ax=ax[0,1], vmin=0, vmax=100)  # Рисуем тепловую карту совпадений по колонкам
    ax[0,1].set_title("Match %")  # Устанавливаем заголовок тепловой карты
    num_cols = [k for k in stats.keys() if "top" not in k][:1]  # Выбираем первую доступную числовую колонку
    if num_cols:  # Проверяем наличие числовых данных для отрисовки
        _sns.kdeplot(data=_pd.DataFrame({num_cols[0]: stats[num_cols[0]]}), ax=ax[1,0])  # Строим наложение распределений (KDE)
        ax[1,0].set_title(f"Dist: {num_cols[0]}")  # Устанавливаем заголовок распределения
    else: ax[1,0].text(0.5, 0.5, "No numeric", ha="center")  # Выводим текстовую заглушку при отсутствии чисел
    ax[1,1].axis("off")  # Отключаем координатные оси для таблицы
    tbl = _pd.DataFrame(list(stats.items()), columns=["Key", "Val"]).style.background_gradient(cmap="Blues")  # Создаём стилизованную DataFrame-таблицу
    ax[1,1].table(cellText=tbl.data.values[:8], colLabels=["Key", "Val"], loc="center")  # Рендерим сводную таблицу на холсте
    _plt.suptitle("Oracle vs PostgresPro")  # Устанавливаем общий заголовок дашборда
    _plt.show()  # Выводим все графики в ячейку блокнота

In [ ]:
def compare(schema_ora: str, schema_pg: str, tables: List[str], cfg_ora: Dict, cfg_pg: Dict, key_col: Optional[str] = None, filters: Optional[Dict] = None, max_rows: int = 5_000_000, chunk: int = 100000):  # Точка входа сравнения
    assert len(tables) == 2, "Требуется ровно 2 таблицы для попарного сравнения"  # Валидируем ограничение на количество таблиц
    strat = CompStrategy()  # Инициализируем стратегию с дефолтными правилами
    cnts, diffs, cache, checked = {"ora": 0, "pg": 0}, {}, {}, 0  # Инициализируем словари и счётчики аккумуляции
    with _ora_conn(cfg_ora) as c_ora, _pg_conn(cfg_pg) as c_pg:  # Открываем оба соединения в едином контексте
        _logger.info("Resolving keys & metadata...")  # Логируем этап подготовки метаданных
        k_ora = key_col or _resolve_pk(c_ora, "ora", schema_ora, tables[0])  # Определяем ключ Oracle (явный или авто-PK)
        k_pg = key_col or _resolve_pk(c_pg, "pg", schema_pg, tables[1])  # Определяем ключ Postgres (явный или авто-PK)
        if k_ora != k_pg: _logger.warning(f"Key mismatch: {k_ora} vs {k_pg}. Ensure semantic equivalence.")  # Предупреждаем о различии имён ключей
        it1, it2 = _stream_chunks(c_ora, "ora", schema_ora, tables[0], k_ora, chunk, filters), _stream_chunks(c_pg, "pg", schema_pg, tables[1], k_pg, chunk, filters)  # Создаём генераторы потоковой выгрузки
        total_est = max_rows // chunk  # Оцениваем количество итераций для прогресс-бара
        for d1, d2 in _tqdm(zip(it1, it2), desc="Comparing chunks", total=total_est):  # Итерируем чанки с визуализацией прогресса
            res = strat.run(d1, d2, k_ora, cache)  # Выполняем сравнение текущих пакетов данных
            cnts["ora"], cnts["pg"] = cnts["ora"] + len(d1), cnts["pg"] + len(d2)  # Обновляем агрегированные счётчики строк
            checked += res["total"]  # Увеличиваем общий счётчик проверенных записей
            for c, n in res["diffs"].items(): diffs[c] = diffs.get(c, 0) + n  # Суммируем расхождения по колонкам
            if checked >= max_rows:  # Проверяем достижение лимита строк
                _logger.info(f"Limit {max_rows} reached. Stopping early.")  # Логируем раннюю остановку
                break  # Прерываем цикл для защиты ресурсов
            del d1, d2, res  # Явно очищаем память от обработанных чанков
        match = {k: round((1 - v/checked)*100, 2) if checked else 100.0 for k, v in diffs.items()}  # Рассчитываем итоговые проценты совпадений
        _draw_dashboard(cnts, _pd.DataFrame([match]), cache)  # Генерируем финальный дашборд
    return {"counts": cnts, "match_pct": match, "stats": cache}  # Возвращаем итоговый словарь-отчёт

In [ ]:
# 📦 КОНФИГУРАЦИЯ И ЗАПУСК
ORA_CFG = {"user": "app_user", "password": "secure_pwd", "dsn": "ora_host:1521/ORCL"}  # Словарь подключения к Oracle
PG_CFG = {"host": "pg_host", "port": 5432, "dbname": "analytics", "user": "pg_user", "password": "secure_pwd"}  # Словарь подключения к PostgresPro

# ✅ Вариант 1: Сравнение с фильтром (рекомендуется для фокусировки на подмножестве)
report_filtered = compare(
    schema_ora="LEGACY_SRC", schema_pg="public", 
    tables=["orders", "orders_replica"], 
    cfg_ora=ORA_CFG, cfg_pg=PG_CFG, 
    filters={"status": "active", "region": "EU"},  # Передаём словарь WHERE-условий
    max_rows=5_000_000
)

# ✅ Вариант 2: Сравнение всей таблицы без фильтров (ключ подберётся автоматически из PK)
report_full = compare(
    schema_ora="LEGACY_SRC", schema_pg="target_db", 
    tables=["customers", "clients"], 
    cfg_ora=ORA_CFG, cfg_pg=PG_CFG, 
    filters=None,  # Явно отключаем фильтрацию для полного скана
    max_rows=5_000_000
)

# 💡 Результаты сохраняются в переменные report_filtered / report_full
# Доступ: report_full["counts"], report_full["match_pct"], report_full["stats"]

## 📊 Примечания к эксплуатации
| Аспект | Реализация | Почему это важно для 5M+ строк |
|--------|------------|--------------------------------|
| **Разные схемы** | `schema_ora` и `schema_pg` передаются раздельно. SQL генерируется независимо. | Позволяет сравнивать legacy-схемы с modern-схемами без переименований в БД. |
| **5 млн строк** | `chunk=100_000` + `Keyset Pagination` (`WHERE key > :val`). `tqdm` показывает прогресс. | Время выборки стабильно `~0.2s/чанк`. RAM не растёт. Нет `OFFSET`-деградации. |
| **Фильтры опциональны** | `filters=None` → `WHERE 1=1`. `filters={"k":"v"}` → безопасная параметризация. | Гибкость: сравнивайте целиком или только `WHERE year=2024 AND status='OK'`. |
| **Безопасность** | Bind-variables, `call_timeout=60s`, `try/finally`, graceful `break`. | Защита от SQLi, зависаний сессий и утечек соединений при обрыве сети. |
| **Расширение** | Добавьте `export_to_parquet(report)` или замените `CompStrategy` на хэш-валидацию. | Архитектура открыта. Паттерн Стратегия позволяет менять логику без правки ядра. |

💡 **Запуск:** Запустите ячейки 1–8 последовательно, затем ячейку 9. Для таблиц >10M строк увеличьте `chunk` до 200 000 и убедитесь, что `key_col` имеет `B-Tree` индекс в обеих БД.